# DealScout - Phase 1, Step 1: Data Curation

Scrub and curate the Amazon product dataset into Item Objects.

Dataset: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.notebook import tqdm
import numpy as np
import random

from dealscout.core.items import Item
from dealscout.core.parser import parse
from dealscout.core.loaders import ItemLoader

load_dotenv(override=True)
login(os.environ["HF_TOKEN"], add_to_git_credential=True)

In [ ]:
dataset_names = [
    "Automotive",
    "Electronics",
    "Office_Products",
    "Tools_and_Home_Improvement",
    "Cell_Phones_and_Accessories",
    "Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
]

items = []
for dataset_name in dataset_names:
    loader = ItemLoader(dataset_name)
    items.extend(loader.load())

print(f"A grand total of {len(items):,} items")

- Removes duplicates by title
- Removes duplicates by full content

In [ ]:
random.seed(42)
random.shuffle(items)

seen = set()
items = [x for x in tqdm(items) if not (x.title in seen or seen.add(x.title))]

seen = set()
items = [x for x in tqdm(items) if not (x.full in seen or seen.add(x.full))]

del seen
print(f"After deduplication, we have {len(items):,} items")

### Weighted Sampling for Dataset Reduction

This code creates a smaller, representative subset of the dataset by selecting **820,000 items** using weighted random sampling. Items are sampled with probability proportional to their **price squared** (`p**2`), meaning higher-priced items are more likely to be included. Two categories are explicitly down-weighted: `Tools_and_Home_Improvement` (50% reduction) and `Automotive` (95% reduction) to prevent them from dominating the sample. The random seed is fixed at `42` for reproducibility, and sampling is done **without replacement** so each item appears at most once. After deduplication, this weighted sample becomes your final working dataset for training or analysis.

In [ ]:
np.random.seed(42)

SIZE = 820_000

prices = np.array([it.price for it in items], dtype=float)
categories = np.array([it.category for it in items])
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)

w = p**2
w[categories == "Tools_and_Home_Improvement"] *= 0.5
w[categories == "Automotive"] *= 0.05

w = w / w.sum()
idx = np.random.choice(len(items), size=SIZE, replace=False, p=w)
sample = [items[i] for i in idx]


In [ ]:
# Just for good measure, let's shuffle the sample again for the final dataset

random.seed(42)
random.shuffle(sample)

In [ ]:
username = "Ankit-Singh-ai"
full = f"{username}/dealscout_items_raw_full"
lite = f"{username}/dealscout_items_raw_lite"

train = sample[:800_000]
val = sample[800_000:810_000]
test = sample[810_000:]

Item.push_to_hub(full, train, val, test)

train_lite = train[:20_000]
val_lite = val[:1_000]
test_lite = test[:1_000]

Item.push_to_hub(lite, train_lite, val_lite, test_lite)